In [12]:
import requests
import pandas as pd
import time

url = "https://api.fda.gov/drug/ndc.json"

# Get total count
meta = requests.get(url).json()
total = meta["meta"]["results"]["total"]
print(f"Total records: {total}")

# Pull everything in pages of 1000
records = []
skip = 0
limit = 1000

while skip < total:
    params = {"limit": limit, "skip": skip}
    response = requests.get(url, params=params)
    data = response.json()
    
    for item in data.get("results", []):
        # Safely grab first active ingredient
        ingredients = item.get("active_ingredients") or []
        first_ingredient = ingredients[0] if ingredients else {}
        
        # Safely grab first package
        packaging = item.get("packaging") or []
        first_package = packaging[0] if packaging else {}
        
        # Flatten the fields we want
        flat = {
            "product_ndc": item.get("product_ndc"),
            "generic_name": item.get("generic_name"),
            "brand_name": item.get("brand_name"),
            "labeler_name": item.get("labeler_name"),
            "dosage_form": item.get("dosage_form"),
            "route": ", ".join(item.get("route") or []),
            "marketing_category": item.get("marketing_category"),
            "listing_expiration_date": item.get("listing_expiration_date"),
            "product_type": item.get("product_type"),
            "finished": item.get("finished"),
            "active_ingredient": first_ingredient.get("name"),
            "strength": first_ingredient.get("strength"),
            "package_ndc": first_package.get("package_ndc"),
            "rxcui": ", ".join(item.get("openfda", {}).get("rxcui") or []),
        }
        records.append(flat)
    
    skip += limit
    time.sleep(0.2)  # polite pause
    print(f"Pulled {min(skip, total)} / {total}")

df = pd.DataFrame(records)
print(f"\nFinal row count: {len(df)}")
print(df.head(3))

Total records: 138229
Pulled 1000 / 138229
Pulled 2000 / 138229
Pulled 3000 / 138229
Pulled 4000 / 138229
Pulled 5000 / 138229
Pulled 6000 / 138229
Pulled 7000 / 138229
Pulled 8000 / 138229
Pulled 9000 / 138229
Pulled 10000 / 138229
Pulled 11000 / 138229
Pulled 12000 / 138229
Pulled 13000 / 138229
Pulled 14000 / 138229
Pulled 15000 / 138229
Pulled 16000 / 138229
Pulled 17000 / 138229
Pulled 18000 / 138229
Pulled 19000 / 138229
Pulled 20000 / 138229
Pulled 21000 / 138229
Pulled 22000 / 138229
Pulled 23000 / 138229
Pulled 24000 / 138229
Pulled 25000 / 138229
Pulled 26000 / 138229
Pulled 27000 / 138229
Pulled 28000 / 138229
Pulled 29000 / 138229
Pulled 30000 / 138229
Pulled 31000 / 138229
Pulled 32000 / 138229
Pulled 33000 / 138229
Pulled 34000 / 138229
Pulled 35000 / 138229
Pulled 36000 / 138229
Pulled 37000 / 138229
Pulled 38000 / 138229
Pulled 39000 / 138229
Pulled 40000 / 138229
Pulled 41000 / 138229
Pulled 42000 / 138229
Pulled 43000 / 138229
Pulled 44000 / 138229
Pulled 45000 / 1382

In [13]:
#Create 11 digit NDC in FDA Dataset

def to_ndc11(ndc, hyphenated=False):
    """Convert package_ndc to standard 11-digit (5-4-2) format."""
    if not ndc or not isinstance(ndc, str):
        return None

    ndc = ndc.strip()
    parts = ndc.split("-")

    # Already unhyphenated 11-digit
    if len(parts) == 1 and len(ndc) == 11 and ndc.isdigit():
        parts = [ndc[:5], ndc[5:9], ndc[9:]]
    elif len(parts) != 3 or not all(p.isdigit() for p in parts):
        return None

    lengths = tuple(len(p) for p in parts)
    if lengths not in {(4, 4, 2), (5, 3, 2), (5, 4, 1), (5, 4, 2)}:
        return None

    labeler = parts[0].zfill(5)
    product = parts[1].zfill(4)
    package = parts[2].zfill(2)

    if hyphenated:
        return f"{labeler}-{product}-{package}"
    return labeler + product + package

# Apply it
df["ndc11"] = df["package_ndc"].apply(lambda x: to_ndc11(x) if pd.notna(x) else None)

In [14]:
#Run this to build a column for status on Package NDC

df["package_ndc_status"] = df["package_ndc"].apply(
    lambda x: "Available" if pd.notna(x) else "Package not available"
)

In [15]:
#Run to expand for dataset
pd.set_option("display.max_colwidth", 40)
df.head(15)

,product_ndc,generic_name,brand_name,labeler_name,dosage_form,route,marketing_category,listing_expiration_date,product_type,finished,active_ingredient,strength,package_ndc,rxcui,ndc11,package_ndc_status
0,76162-119,Polyethylene Glycol 3350,Topcare ClearLax,Topco Associates LLC,"POWDER, FOR SOLUTION",ORAL,ANDA,20271231,HUMAN OTC DRUG,True,POLYETHYLENE GLYCOL 3350,17 g/17g,76162-119-02,"876193, 880407",76162011902,Available
1,76282-247,Levetiracetam,Levetiracetam,"Exelan Pharmaceuticals, Inc.",TABLET,ORAL,ANDA,20271231,HUMAN PRESCRIPTION DRUG,True,LEVETIRACETAM,500 mg/1,76282-247-12,"311288, 311289, 311290",76282024712,Available
2,76348-495,"Menthol, Methyl Salicylate",EXTREME EMU,"Renu Laboratories, LLC",CREAM,TOPICAL,OTC MONOGRAPH DRUG,20261231,HUMAN OTC DRUG,True,METHYL SALICYLATE,11.2 g/112g,76348-495-04,311503,76348049504,Available
3,76385-112,Carvedilol,Carvedilol,"UNICHEM PHARMACEUTICALS (USA), INC.","TABLET, FILM COATED",ORAL,ANDA,20271231,HUMAN PRESCRIPTION DRUG,True,CARVEDILOL,12.5 mg/1,76385-112-01,"200031, 200032, 200033, 686924",76385011201,Available
4,0002-7511,Insulin lispro,Humalog Mix75/25,Eli Lilly and Company,"INJECTION, SUSPENSION",SUBCUTANEOUS,BLA,20271231,HUMAN PRESCRIPTION DRUG,True,INSULIN LISPRO,100 [iU]/mL,0002-7511-01,"259111, 752388, 847252, 847254",00002751101,Available
5,76420-674,PANTOPRAZOLE,PANTOPRAZOLE SODIUM,"Asclemed USA, Inc.","TABLET, DELAYED RELEASE",ORAL,ANDA,20271231,HUMAN PRESCRIPTION DRUG,True,PANTOPRAZOLE SODIUM,40 mg/1,76420-674-01,"251872, 314200",76420067401,Available
6,0597-0405,adalimumab-adbm,Cyltezo,Boehringer Ingelheim Pharmaceuticals...,KIT,,BLA,20271231,HUMAN PRESCRIPTION DRUG,True,NaN,NaN,0597-0405-80,"797544, 2640883, 2640889, 2640891, 2...",00597040580,Available
7,76891-155,Alcohol,Hand Sanitizer Matcha Mood,Scent Theory Products LLC,SPRAY,TOPICAL,OTC MONOGRAPH DRUG,20261231,HUMAN OTC DRUG,True,ALCOHOL,62 mL/100mL,76891-155-30,1020364,76891015530,Available
8,77931-0001,"Fucus Vesiculosus, Chelidonium Majus...",Drop20,Pick Enterprises Corp,LIQUID,ORAL,UNAPPROVED HOMEOPATHIC,20271231,HUMAN OTC DRUG,True,FUCUS VESICULOSUS,4 [hp_X]/mL,77931-0001-1,,77931000101,Available
9,79794-189,antibacterial hand wipes,antibacterial hand wipes (Cherry),Zhejiang Furuisen Spunlaced Nonwoven...,CLOTH,CUTANEOUS,OTC MONOGRAPH DRUG,20271231,HUMAN OTC DRUG,True,BENZALKONIUM CHLORIDE,.13 g/1,79794-189-01,1038558,79794018901,Available


In [16]:
import pandas as pd
import zipfile
import requests
from pathlib import Path

url = "https://download.nlm.nih.gov/rxnorm/RxNorm_full_prescribe_11022026.zip"
zip_path = Path("RxNorm_full_prescribe_11022026.zip")

# Download if not already present
if not zip_path.exists():
    print("Downloading...")
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)
    print("Download complete.")

# Extract NDC crosswalk
with zipfile.ZipFile(zip_path) as z:
    sat = pd.read_csv(
        z.open("rrf/RXNSAT.RRF"),
        sep="|",
        header=None,
        dtype=str,
        usecols=[0, 8, 9, 10],          # RXCUI, ATN, SAB, ATV
        names=["rxcui", "atn", "sab", "ndc11"]
    )

ndc_xwalk = (
    sat[(sat["atn"] == "NDC") & (sat["sab"] == "RXNORM")]
    [["ndc11", "rxcui"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(ndc_xwalk.shape)
print(ndc_xwalk.head())
print("Unique NDCs:", ndc_xwalk["ndc11"].nunique())
print("Unique RxCUIs:", ndc_xwalk["rxcui"].nunique())

# Optional: save it
ndc_xwalk.to_parquet("rxnorm_ndc_xwalk.parquet", index=False)
# or
# ndc_xwalk.to_csv("rxnorm_ndc_xwalk.csv", index=False)

(239693, 2)
         ndc11  rxcui
0  59050026800  91348
1  00295117904  91349
2  00295117916  91349
3  00363026816  91349
4  00363026832  91349
Unique NDCs: 239693
Unique RxCUIs: 16713


In [8]:
import pandas as pd
import requests
from pathlib import Path

# Paths
parquet_path = Path(r"C:\Users\Patri\OneDrive\Desktop\NDC lookup Table\rxnorm_ndc_xwalk.parquet")
out_path = Path(r"C:\Users\Patri\OneDrive\Desktop\NDC lookup Table\rxnorm_ndc_xwalk_with_names.xlsx")

# Load the crosswalk
df = pd.read_parquet(parquet_path)
print(f"Loaded {len(df):,} rows")

# Get unique RxCUIs
rxcuis = df["rxcui"].dropna().unique()
print(f"Unique RxCUIs: {len(rxcuis):,}")

# Function to pull the preferred name from RxNav
def get_name(rxcui):
    try:
        url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/properties.json"
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            return data.get("properties", {}).get("name")
    except Exception:
        pass
    return None

# Pull names (this will take a few minutes)
print("Fetching names from RxNav...")
name_map = {}
for i, rxcui in enumerate(rxcuis):
    name_map[rxcui] = get_name(rxcui)
    if (i + 1) % 500 == 0:
        print(f"  {i + 1:,} / {len(rxcuis):,}")

# Map back and save
df["rxnorm_name"] = df["rxcui"].map(name_map)
df.to_excel(out_path, index=False)
print(f"Saved to {out_path}")

Loaded 239,693 rows
Unique RxCUIs: 16,713
Fetching names from RxNav...


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

# Load your unique RxCUIs (adjust path if needed)
xwalk = pd.read_parquet(r"C:\Users\Patri\OneDrive\Desktop\NDC lookup Table\rxnorm_ndc_xwalk.parquet")
rxcuis = xwalk["rxcui"].dropna().unique().tolist()
print(f"Unique RxCUIs: {len(rxcuis):,}")

def fetch_concept(rxcui):
    url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/properties.json"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            data = r.json()
            prop = data.get("properties", {})
            return {
                "rxcui": rxcui,
                "name": prop.get("name"),
                "tty": prop.get("tty"),
                "suppress": prop.get("suppress"),
            }
    except Exception:
        pass
    return {"rxcui": rxcui, "name": None, "tty": None, "suppress": None}

print("Fetching concepts from RxNav (parallel)...")
results = []
with ThreadPoolExecutor(max_workers=8) as executor:  # 8 workers — increase carefully if it stays stable
    futures = {executor.submit(fetch_concept, rxcui): rxcui for rxcui in rxcuis}
    for i, future in enumerate(as_completed(futures), 1):
        results.append(future.result())
        if i % 500 == 0:
            print(f"  {i:,} / {len(rxcuis):,}")

concepts = pd.DataFrame(results)
print(f"Done. Concepts retrieved: {len(concepts):,}")
print(concepts.head())

# Optional: save it so you don't have to re-fetch
out_path = Path(r"C:\Users\Patri\OneDrive\Desktop\NDC lookup Table\rxnav_concepts.parquet")
concepts.to_parquet(out_path, index=False)
print(f"Saved to {out_path}")

In [3]:
import pandas as pd
from pathlib import Path

output_folder = Path(r"C:\Users\Patri\OneDrive\Desktop\NDC lookup Table")
output_folder.mkdir(parents=True, exist_ok=True)

output_file = output_folder / "rxcui_concepts.xlsx"

# Convert the list to a DataFrame, then save
df = pd.DataFrame(results)
df.to_excel(output_file, index=False)

print(f"Saved: {output_file}")
print(f"Rows: {len(df):,}")

Saved: C:\Users\Patri\OneDrive\Desktop\NDC lookup Table\rxcui_concepts.xlsx
Rows: 16,713


In [5]:
# Adjust the left_on / right_on to the actual column names
joined = df.merge(
    ndc_xwalk,
    left_on="NDC",          # change to the real column in DF
    right_on="ndc",         # change to the real column in ndc_xwalk
    how="left"
)

print(f"Joined shape: {joined.shape}")
joined.head(10)

NameError: name 'ndc_xwalk' is not defined